In [152]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from ITIS_Model_funcs import get_nominal_param,OLS_res,ITIS

In [153]:
param_log,IC = get_nominal_param()
IC = [ 2.0,          0.,          0.14759607,  0.,         13.75484703,  3.13344442, 15.73028492,  1.38649737]

D = '3'         # '1' for ACTH + Cort,
                # '2' for ACTH + Cort + TNF-a
                # '3' for ACTH + Cort + TNF-a + IL10
dpoints = '2'   # '1' for 25, '2' for 13

if D == '1':
    output_ids = [6,7]
elif D == '2':
    output_ids = [3,6,7]
else:
    output_ids = [3,4,6,7]

with open('syntheticData\\TRUE_SOL' + D + dpoints + '.pkl', 'rb') as f:
    results = pickle.load(f)

t_data = results['t_data']
y_data = results['y_data']

##SENSITIVITY ANALYSIS
h = 1e-6  #amount to perturb parameters
n_param = len(param_log)
n_states = len(output_ids)

S = np.zeros((n_param, len(t_data) * n_states)) ##Initialize shape of sensitivity matrix.

for i in range(n_param):  #calculate the relative residual sensitivity to each 45 parameters

        param_in = param_log[i]
        param_delta = param_in + h
##RIGHT NOW THIS IS BEING CALCULATED WITH NOISELESS DATA
        S[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_log, IC)
                                            - OLS_res(param_in, y_data, t_data, i, output_ids, param_log, IC)))

In [154]:
##import rankings from global analysis
with open('paramRankings\\rankingDesign' + D + '.pkl', 'rb') as f:
    results = pickle.load(f)

rank_value = results['rank_value']      # sorted ranking values
param_sorted = results['param_sorted']  #accordingly sorted params as strings
rank_cut = np.array(rank_value)[np.array(rank_value) > .25 * np.array(rank_value)[0]]
param_sorted = np.array(param_sorted)[:len(rank_cut)]

param_titles = ['d1',
'k1','k2','h1','h2','h3','d2',
'k3','k4','h4','d3',
'h5','h6','k5','k6','h7','d4',
'b1','k7','h8','k8','h9','d5','h10',
'b2','k9','k10','k11','d6',
'k12','k13','k14','h11','d7',
'k15','k16','d8',
'alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc']

circadian_param = ['alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc'] #exclude?

cond = 0
unid = []
p_indices = [param_titles.index(param_sorted[0])]
for i in range(1,len(rank_cut)):
    p_indices.append(param_titles.index(param_sorted[i]))
    S_opt = S[p_indices,:].copy()
    F_opt = S_opt@S_opt.T
    cond = np.linalg.cond(F_opt)
    if cond > 1e+5:
        unid.append(p_indices.pop())
        print("Removed: ", unid[-1])


S_opt = S[p_indices,:].copy()
F_opt = S_opt@S_opt.T
C_opt = np.linalg.inv(F_opt)
selected = [param_titles[i] for i in p_indices]
print("Design : ", D + dpoints)
print("Number of selected parameters:", len(p_indices))
print("Selected Parameters: ",  selected)
print(p_indices)
print("Excluded Parameters: ", [param_titles[i] for i in unid])
print("Condition Number of F:", np.linalg.cond(F_opt))
print("Diagonals of C:", np.diag(C_opt))

Design :  32
Number of selected parameters: 16
Selected Parameters:  ['h6', 'd7', 'd4', 'k3', 'T', 'd8', 'beta', 'h7', 'h11', 'k14', 'k4', 'h9', 'h4', 'k6', 'd3', 'k15']
[12, 33, 16, 7, 43, 36, 39, 15, 32, 31, 8, 21, 9, 14, 10, 34]
Excluded Parameters:  []
Condition Number of F: 18532.770503143198
Diagonals of C: [0.11773081 0.57199983 0.27221793 0.05694667 0.05835943 0.12148096
 0.10348279 0.10057655 0.90165451 0.10962141 0.1636426  0.03793812
 0.61277424 0.52360455 0.1119123  0.1055435 ]


In [155]:
#Complete F
F = S@S.T
C = np.linalg.inv(F)
print("Condition Number of F:", np.linalg.cond(F))
print("Diagonals of C:", np.diag(C))
diag = np.diag(C)
unid = [param_titles[i] for i in list(np.where(diag > 1e+3))[0]]
print("Unidentifiable:", unid) ##this isn't really useful right now

Condition Number of F: 58507862069.57545
Diagonals of C: [2.95387196e+03 6.46978749e+03 4.46676358e+03 4.29879142e+01
 4.78466539e+02 2.26575517e+02 9.71943019e+01 6.05106105e+02
 9.67842880e+01 1.24940927e+03 5.46389568e+02 5.87314246e+03
 4.80439309e+02 6.81389751e+04 1.34147165e+04 6.20855856e+02
 2.51553630e+02 1.86699573e+04 2.97180887e+02 7.39919768e+02
 4.24870856e+02 5.46304040e+01 4.65125710e+02 3.93306486e+01
 1.17079980e+02 2.27315156e+04 3.30218642e+02 1.12355636e+03
 4.27640563e+03 8.49870903e+03 1.67383416e+03 5.37658040e+03
 5.98849085e+03 9.96001250e+03 1.39465696e+01 7.93941838e+01
 1.31236750e+02 6.98580525e+03 2.13394036e+06 2.92675364e+03
 1.30328825e+03 4.62950143e+03 1.53113746e+04 5.96107407e+00
 2.01007257e+01]
Unidentifiable: ['d1', 'k1', 'k2', 'h4', 'h5', 'k5', 'k6', 'b1', 'k9', 'k11', 'd6', 'k12', 'k13', 'k14', 'h11', 'd7', 'alpha', 'k', 'beta', 'L', 'eps', 'delta']


In [156]:
##SAVE FUNCTIONALITY

all_results = {
    'S': S,
    'S_opt': S_opt,
}

with open('Sensitivities\\sens' + D + dpoints + '.pkl', 'wb') as f:
    pickle.dump(all_results, f)